## Imports

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:

!pip install -q biopython==1.81 peft==0.10.0 "transformers>=4.48,<4.56" "tokenizers>=0.21,<0.22"\
    accelerate==0.29.3 sentencepiece==0.2.0 huggingface_hub safetensors evaluate rouge_score tqdm graphein bert_score torch-geometric sentence-transformers --upgrade torchao


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 3.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 6.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 7.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 7.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.2/47.2 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.1/199.1 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 297.6/297.6 kB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 77.4 MB/s

In [ ]:
!pip install --upgrade numpy --upgrade pandas

In [ ]:
!wget https://mmseqs.com/foldseek/foldseek-linux-avx2.tar.gz
!tar xvzf foldseek-linux-avx2.tar.gz

# Add foldseek binary path to your environment session
import os
os.environ["PATH"] += ":/content/foldseek/bin"

--2026-08-01 11:14:46--  https://mmseqs.com/foldseek/foldseek-linux-avx2.tar.gz
Resolving mmseqs.com (mmseqs.com)... 158.247.200.62
Connecting to mmseqs.com (mmseqs.com)|158.247.200.62|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 11627535 (11M) [application/octet-stream]
Saving to: ‘foldseek-linux-avx2.tar.gz.4’

foldseek-linux-avx2 100%[===================>]  11.09M  13.8MB/s    in 0.8s    

2026-08-01 11:14:48 (13.8 MB/s) - ‘foldseek-linux-avx2.tar.gz.4’ saved [11627535/11627535]

foldseek/
foldseek/README.md
foldseek/bin/
foldseek/bin/foldseek


In [ ]:
import os, sys, json, random, re, time

import numpy as np
import pandas as pd
import torch

repo_root   = "/content/drive/MyDrive/Colab_Notebooks/266_Final_Project/Prot2Text-V2-main"
esm_path    = "facebook/esm2_t36_3B_UR50D"
llama_path  = "meta-llama/Llama-3.1-8B-Instruct"
hf_model_id = "xiao-fei/Prot2Text-V2-11B-Instruct-hf"

csv_dir  = "/content/drive/MyDrive/Colab_Notebooks/266_Final_Project/Prot2TextDataset"
work_dir = "/data/foldseek_work"
out_dir  = "/data/runs/foldseek_lora"

random_seed = 0
splits = ["train", "eval", "test"]

os.makedirs(work_dir, exist_ok=True)
os.makedirs(out_dir, exist_ok=True)
random.seed(random_seed)
torch.manual_seed(random_seed)

print("work:", work_dir, "\nout :", out_dir)
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    _p = torch.cuda.get_device_properties(0)
    print(f"gpu: {_p.name}  {_p.total_memory/1e9:.1f} GB")

work: /data/foldseek_work 
out : /data/runs/foldseek_lora
torch 2.11.0+cu128 | cuda True
gpu: NVIDIA A100-SXM4-80GB  85.1 GB


In [ ]:
# CSV columns used by Prot2TextLightDataset: AlphaFoldDB, Full Name, taxon, sequence, function
raw = {}
for s in SPLITS:
    raw[s] = pd.read_csv(os.path.join(csv_dir, f"{s}.csv"))
    print(f"{s:5s} n={len(raw[s]):>7,}  cols={list(raw[s].columns)}")

train n=248,315  cols=['accession', 'name', 'Full Name', 'taxon', 'sequence', 'function', 'AlphaFoldDB']
eval  n=  4,172  cols=['accession', 'name', 'Full Name', 'taxon', 'sequence', 'function', 'AlphaFoldDB']
test  n=  4,203  cols=['accession', 'name', 'Full Name', 'taxon', 'sequence', 'function', 'AlphaFoldDB']


## Part 1 — Alphafolddb structures and Foldseek retrieval



In [ ]:
pdb_root = os.path.join(work_dir, "alphafold_pdbs")   # one subdirectory per split
alphafold_file_template = "https://alphafold.ebi.ac.uk/files/AF-{acc}-F1-model_v4.pdb"
alphafold_api_template  = "https://alphafold.ebi.ac.uk/api/prediction/{acc}"
download_workers  = 16      # concurrent HTTP requests
download_retries  = 2      # attempts beyond the first, per accession (network errors only)
download_timeout = 15     # seconds per request
download_backoff  = 0.5    # seconds between retries
progress_every    = 2000    # print a progress line every N completed downloads

import urllib.error
import urllib.request
from concurrent.futures import ThreadPoolExecutor, as_completed


json_headers = {"User-Agent": "prot2text-foldseek/1.0", "Accept": "application/json"}
file_headers = {"User-Agent": "prot2text-foldseek/1.0"}


def _fetch_json(url, timeout=download_timeout):
  '''obtain json file from url
  '''
    req = urllib.request.Request(url, headers=json_headers)
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        return json.loads(resp.read().decode("utf-8"))


def _fetch_bytes(url, timeout=download_timeout):
  '''obtain raw bytes from url- needed for pdb files
  '''
    req = urllib.request.Request(url, headers=file_headers)
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        return resp.read()


def _download_one(acc, out_dir):
    """
    Fetch one AlphaFoldDB structure. Returns accession where status is "ok", "skip", or "fail".
    Creates out_dir if it doesn't exist. Deals with various download errors
    """
    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, f"{acc}.pdb")
    if os.path.exists(out_path) and os.path.getsize(out_path) > 0:
        return acc, "skip"

    acc_u = acc.strip().upper()
    direct_url = alphafold_file_template.format(acc=acc_u)

    for attempt in range(download_retries + 1):
        try:
            data = _fetch_bytes(direct_url)
            with open(out_path, "wb") as fh:
                fh.write(data)
            return acc, "ok"
        except urllib.error.HTTPError as e:
            if e.code == 404:
                break
            if attempt < download_retries:
                time.sleep(download_backoff)
        except Exception:
            if attempt < download_retries:
                time.sleep(download_backoff)

    # Fallback: resolve the current URL via the prediction API. Covers entries
    # added or changed since the direct v4 pattern was last valid for them.
    try:
        meta = _fetch_json(alphafold_api_template.format(acc=acc_u))
        if not meta:
            return acc, "fail"          # accession has no AlphaFoldDB entry
        pdb_url = meta[0].get("pdbUrl")
        if not pdb_url:
            return acc, "fail"
        data = _fetch_bytes(pdb_url)
        if not data:
            return acc, "fail"
        with open(out_path, "wb") as fh:
            fh.write(data)
        return acc, "ok"
    except Exception:
        return acc, "fail"


def download_split_structures(split, accessions, out_dir):
    """Download AlphaFoldDB structures for all accessions in one split. Returns the list of accessions
    that failed (no structure available or all retries exhausted)."""
    os.makedirs(out_dir, exist_ok=True)
    accessions = [a for a in accessions if isinstance(a, str)]
    n_total = len(accessions)
    n_ok = n_skip = n_fail = 0
    failed = []
    t0 = time.time()

    with ThreadPoolExecutor(max_workers=download_workers) as ex:
        futures = {ex.submit(_download_one, acc, out_dir): acc for acc in accessions}
        for i, fut in enumerate(as_completed(futures), 1):
            acc, status = fut.result()
            if status == "ok":
                n_ok += 1
            elif status == "skip":
                n_skip += 1
            else:
                n_fail += 1
                failed.append(acc)
            if i % progress_every == 0 or i == n_total:
                elapsed = time.time() - t0
                rate = i / elapsed if elapsed > 0 else 0.0
                remaining = (n_total - i) / rate if rate > 0 else 0.0
                eh, rem = divmod(int(remaining), 3600)
                em, es = divmod(rem, 60)
                print(
                    f"  [{split}] {i:>7,}/{n_total:,} ({i/n_total:5.1%})"
                    f"  ok={n_ok:,} skip={n_skip:,} fail={n_fail:,}"
                    f"  {rate:.1f}/s  eta {eh}h{em:02d}m{es:02d}s"
                )

    elapsed = time.time() - t0
    eh, rem = divmod(int(elapsed), 3600)
    em, es = divmod(rem, 60)
    print(
        f"  [{split}] done in {eh}h{em:02d}m{es:02d}s: "
        f"{n_ok:,} downloaded, {n_skip:,} already present, {n_fail:,} failed "
        f"({n_fail/max(n_total,1):.1%} of {n_total:,})"
    )
    return failed

#Run alphafold downloads for the train, eval, test split
pdb_dirs = {}
failed_accessions = {}
for s in splits:
    pdb_dirs[s] = os.path.join(pdb_root, s)
    failed_accessions[s] = download_split_structures(
        s, raw[s]["AlphaFoldDB"], pdb_dirs[s]
    )
# Save failed-accession lists. Proteins that failed get no retrieval hits
fail_path = os.path.join(work_dir, "failed_downloads.json")
with open(fail_path, "w") as fh:
    json.dump(failed_accessions, fh, indent=2)
print("\nfailed-accession lists saved to", fail_path)

  [train]   2,000/248,315 ( 0.8%)  ok=2,000 skip=0 fail=0  121.9/s  eta 0h33m40s
  [train]   4,000/248,315 ( 1.6%)  ok=4,000 skip=0 fail=0  136.3/s  eta 0h29m52s
  [train]   6,000/248,315 ( 2.4%)  ok=5,999 skip=0 fail=1  142.3/s  eta 0h28m22s
  [train]   8,000/248,315 ( 3.2%)  ok=7,999 skip=0 fail=1  145.7/s  eta 0h27m28s
  [train]  10,000/248,315 ( 4.0%)  ok=9,999 skip=0 fail=1  147.5/s  eta 0h26m55s
  [train]  12,000/248,315 ( 4.8%)  ok=11,999 skip=0 fail=1  148.8/s  eta 0h26m27s
  [train]  14,000/248,315 ( 5.6%)  ok=13,999 skip=0 fail=1  149.5/s  eta 0h26m07s
  [train]  16,000/248,315 ( 6.4%)  ok=15,999 skip=0 fail=1  150.3/s  eta 0h25m45s
  [train]  18,000/248,315 ( 7.2%)  ok=17,999 skip=0 fail=1  151.0/s  eta 0h25m25s
  [train]  20,000/248,315 ( 8.1%)  ok=19,999 skip=0 fail=1  151.6/s  eta 0h25m05s
  [train]  22,000/248,315 ( 8.9%)  ok=21,999 skip=0 fail=1  152.2/s  eta 0h24m47s
  [train]  24,000/248,315 ( 9.7%)  ok=23,999 skip=0 fail=1  152.7/s  eta 0h24m28s
  [train]  26,000/248

In [ ]:
foldseek_bin = "foldseek"

def which(prog):
    """Locate an executable on path."""
    if os.path.dirname(prog):
        return prog if os.access(prog, os.X_OK) else None
    for d in os.environ.get("PATH", "").split(os.pathsep):
        cand = os.path.join(d, prog)
        if os.path.isfile(cand) and os.access(cand, os.X_OK):
            return cand
    return None

def sh(cmd, check=True):
    """
    Run a shell command and echo its combined output. Needed to interact with foldseek library
    """
    print("$", cmd)
    stream = os.popen("( " + cmd + " ) 2>&1")
    output = stream.read()
    status = stream.close()
    if output:
        print(output[-4000:])
    if check and status is not None:
        raise RuntimeError(f"command failed (status {status}): {cmd}")
    return output


sh(f"{foldseek_bin} version", check=False)

$ foldseek version
7f4b94644d6b210af4530e04a9fdc9e4bb64ea66



'7f4b94644d6b210af4530e04a9fdc9e4bb64ea66\n'

In [ ]:
index_db = os.path.join(work_dir, "trainDB")


#Build foldseek structural index from training data- reads structure data from pdb files
if not os.path.exists(index_db + ".dbtype"):
    train_pdb_dir = pdb_dirs["train"]
    t0 = time.time()
    sh(f'{foldseek_bin} createdb "{train_pdb_dir}" "{index_db}"')
    print(f"createdb took {time.time()-t0:.0f}s")
else:
    print("index already built:", index_db)

$ foldseek createdb "/data/foldseek_work/alphafold_pdbs/train" "/data/foldseek_work/trainDB"
createdb /data/foldseek_work/alphafold_pdbs/train /data/foldseek_work/trainDB 

MMseqs Version:             	7f4b94644d6b210af4530e04a9fdc9e4bb64ea66
Use GPU                     	0
Path to ProstT5             	
Chain name mode             	0
Model name mode             	0
Createdb extraction mode    	0
Interface distance threshold	10
Write mapping file          	0
Write Foldcomp              	0
Mask b-factor threshold     	0
Coord store mode            	2
Write lookup file           	1
Input format                	0
Input compression format    	0
File Inclusion Regex        	.*
File Exclusion Regex        	^$
Threads                     	44
Verbosity                   	3

Output file: /data/foldseek_work/trainDB
[=================================================================] 248.30K 16s 759ms
Time for merging to trainDB_ss: 0h 0m 0s 108ms
Time for merging to trainDB_h: 0h 0m 0s 85ms
Time fo

In [ ]:
fs_sensitivity = 9.5 #1-10. Higher = more time spent searching for remote structural homologs
fs_max_seqs    = 300 #how many hits to retain per query
fs_evalue      = 10.0 #expectation value threshold (significance of matching protein structures)
fs_threads     = 44
fs_format      = "query,target,fident,alnlen,evalue,bits,qcov,tcov"

# Search each split's structures against the training index. All splits are searched:
#   train: to find nearest neighbours
#   eval/test: to find structurally similar training proteins whose
#              function descriptions will populate retrieval prompts
m8 = {}
for s in splits:
    out = os.path.join(work_dir, f"hits_{s}.m8")
    m8[s] = out
    if os.path.exists(out) and os.path.getsize(out) > 0:
        print("exists, skipping:", os.path.basename(out))
        continue
    query_dir = pdb_dirs[s]
    tmp = os.path.join(work_dir, f"tmp_{s}")
    t0 = time.time()
    sh(
        f'{foldseek_bin} easy-search "{query_dir}" "{index_db}" "{out}" "{tmp}"'
        f' --format-output "{fs_format}"'
        f' -s {fs_sensitivity} --max-seqs {fs_max_seqs}'
        f' -e {fs_evalue} --threads {fs_threads}'
    )
    print(f"[{s}] search took {time.time()-t0:.0f}s")

$ foldseek easy-search "/data/foldseek_work/alphafold_pdbs/train" "/data/foldseek_work/trainDB" "/data/foldseek_work/hits_train.m8" "/data/foldseek_work/tmp_train" --format-output "query,target,fident,alnlen,evalue,bits,qcov,tcov" -s 9.5 --max-seqs 300 -e 10.0 --threads 44
=================] 248.30K 0s 123ms
Index table: Masked residues: 9182770
Index table: fill
[=================================================================] 248.30K 0s 441ms
Index statistics
Entries:          74625332
DB size:          915 MB
Avg k-mer size:   1.166021
Top 10 k-mers
    LVLVVV	94400
    SVSVVV	78256
    VVLVVV	71584
    VVSVVV	57552
    LVVVVV	53398
    VLVLLL	47232
    VVVSVS	44805
    VSVSSS	43453
    SVVVVV	40882
    VSVSVS	40283
Time for index table init: 0h 0m 1s 33ms
Process prefiltering step 1 of 1

k-mer similarity threshold: 78
Starting prefiltering scores calculation (step 1 of 1)
Query db start 1 to 248305
Target db start 1 to 248305
[====================================================

In [ ]:
#save foldseek output- allows reloading to avoid keeping pdb files which are no longer needed
!cp {INDEX_DB}* /content/drive/MyDrive/Colab_Notebooks/266_Final_Project/foldseek_index/
!cp /data/foldseek_work/hits_*.m8 /content/drive/MyDrive/Colab_Notebooks/266_Final_Project/foldseek_index/

cp: cannot stat '/data/foldseek_work/hits_*.m8': No such file or directory


In [ ]:
# reload the CSVs (needed for train_desc, taxon, sequence, function)
import shutil
drive_index_dir = "/content/drive/MyDrive/Colab_Notebooks/266_Final_Project/foldseek_index"
out_dir  = "/content/drive/MyDrive/Colab_Notebooks/266_Final_Project/foldseek_lora/drop01run"
work_dir = "/content/drive/MyDrive/Colab_Notebooks/266_Final_Project/foldseek_work"
raw = {}
for s in splits:
    raw[s] = pd.read_csv(os.path.join(csv_dir, f"{s}.csv"))
    print(f"{s:5s} n={len(raw[s]):>7,}")

# restore the foldseek index from drive
index_db = os.path.join(work_dir, "trainDB")
for f in os.listdir(drive_index_dir):
    if f.startswith("trainDB"):
        shutil.copy(os.path.join(drive_index_dir, f), os.path.join(work_dir, f))
assert os.path.exists(index_db + ".dbtype"), "index files did not copy correctly"
print("restored Foldseek index:", index_db)

# restore the hit tables from drive
fs_format = "query,target,fident,alnlen,evalue,bits,qcov,tcov"

m8 = {}
for s in splits:
    dst = os.path.join(work_dir, f"hits_{s}.m8")
    shutil.copy(os.path.join(drive_index_dir, f"hits_{s}.m8"), dst)
    m8[s] = dst
    print(f"restored {s}: {os.path.getsize(dst)/1e6:.1f} MB")

train n=248,315
eval  n=  4,172
test  n=  4,203
restored Foldseek index: /content/drive/MyDrive/Colab_Notebooks/266_Final_Project/foldseek_work/trainDB
restored train: 3064.9 MB
restored eval: 40.2 MB
restored test: 40.2 MB


In [ ]:
# max hits to retain per query protein during parsing
keep_top_hits = 10

cols = fs_format.split(",")

# Build a lookup from uniprot accession to function text for training split.
# When foldseek identifies a training protein as structurally similar to a query,
# its function text is fetched from this dict and placed in the prompt.
train_desc = {
    a: d for a, d in zip(raw["train"]["AlphaFoldDB"], raw["train"]["function"])
    if isinstance(a, str) and isinstance(d, str)
}
print(f"{len(train_desc):,} training descriptions available as retrieval payload")


def parse_hits(path, split, keep_top=keep_top_hits):
    # Parse a foldseek .m8 file into {query_accession: [hit, ...]}, best first. Drops self hits and keeps only top hits
    hits = {}
    dropped_self = 0
    with open(path) as fh:
        for line in fh:
            parts = line.rstrip("\n").split("\t")
            if len(parts) < len(cols):
                continue                     # malformed line
            rec = dict(zip(cols, parts))
            q, t = rec["query"], rec["target"]
            if q == t:
                dropped_self += 1            # self-hit: protein finding itself
                continue
            if t not in train_desc:
                continue                     # no description available
            bucket = hits.setdefault(q, [])
            if len(bucket) >= keep_top:
                continue
            bucket.append({
                "target": t,
                "fident": float(rec["fident"]),  # fractional sequence identity
                "qcov":   float(rec["qcov"]),    # fraction of query covered
                "evalue": float(rec["evalue"]),  # statistical significance
                "bits":   float(rec["bits"]),    # alignment bit score
                "desc":   train_desc[t],         # function text from training CSV
            })
    print(f"{split:5s}: {len(hits):,} queries with >=1 usable hit  (self-hits dropped: {dropped_self:,})")
    return hits


hits = {s: parse_hits(m8[s], s) for s in splits}

# confirm no self-hits survived the filtering above
for s in splits:
    for q, hs in hits[s].items():
        assert all(h["target"] != q for h in hs), f"self-hit survived for {q}"
print("no self-hits remain")

# Build a lookup from training accession to organism name.

hit_taxon_lookup = {
    str(row["AlphaFoldDB"]): str(row["taxon"])
    for _, row in raw["train"].iterrows()
    if isinstance(row["AlphaFoldDB"], str) and isinstance(row["taxon"], str)
}
print(f"taxon lookup: {len(hit_taxon_lookup):,} training accessions")

# verify a sample of retrieved target accessions are in the
# taxon lookup. Missing entries get the fallback label 'unknown organism'.
sample_targets = [
    h["target"]
    for hs in list(hits["test"].values())[:20]
    for h in hs[:1]
]
missing = [t for t in sample_targets if t not in hit_taxon_lookup]
print(f"sample check: {len(sample_targets)} targets, {len(missing)} missing from lookup")
if missing:
    print("  missing examples:", missing[:5])

248,315 training descriptions available as retrieval payload
train: 245,375 queries with >=1 usable hit  (self-hits dropped: 245,265)
eval : 4,128 queries with >=1 usable hit  (self-hits dropped: 0)
test : 4,163 queries with >=1 usable hit  (self-hits dropped: 0)
no self-hits remain
taxon lookup: 248,029 training accessions
sample check: 20 targets, 0 missing from lookup


In [ ]:
# Print the distribution of best-hit sequence identity (fident) across each split
# Hit rate = fraction of proteins in the split with at least one usable hit.
# p10/25/50/75/90 are the fraction of aligned residue positions that are identical between the query and its best structural match
# eval and test hits are lower which is expected
for s in splits:
    best = np.array([hs[0]["fident"] for hs in hits[s].values()]) if hits[s] else np.array([])
    cov  = len(hits[s]) / max(len(raw[s]), 1)
    if best.size:
        qs = np.percentile(best, [10, 25, 50, 75, 90])
        print(f"{s:5s} hit-rate={cov:5.1%}  best-hit fident p10/25/50/75/90 = "
              + " / ".join(f"{v:.2f}" for v in qs))
    else:
        print(f"{s:5s} hit-rate={cov:5.1%}  (no hits)")

train hit-rate=98.8%  best-hit fident p10/25/50/75/90 = 0.39 / 0.61 / 0.83 / 0.96 / 0.99
eval  hit-rate=98.9%  best-hit fident p10/25/50/75/90 = 0.17 / 0.25 / 0.35 / 0.53 / 0.78
test  hit-rate=99.0%  best-hit fident p10/25/50/75/90 = 0.17 / 0.25 / 0.35 / 0.55 / 0.77


### Downsampling Easy Train Hits

In [ ]:
# Identity threshold above which a training hit is 'easy'. Used to minimize proteins that find a near-identical neighbourwhich  would produce prompts where the retrieved
# description is essentially the correct answer
easy_ident_threshold = 0.60
# Probability of keeping an easy training hit. At 0.25, 75% of easy hits are dropped so the model sees them rarely and cannot rely on copying
easy_keep_prob       = 0.25
# Number of hits placed into each prompt
max_hits_in_prompt   = 2
rng = random.Random(random_seed)

def select_for_prompt(hits, split, max_hits=max_hits_in_prompt):
    # Select hits to include in the prompt, applying downsampling to easy training hits. For eval/test: all hits
    # pass through unchanged.
    out = []
    for h in hits:
        if split == "train" and h["fident"] > easy_ident_threshold:
            if rng.random() > easy_keep_prob:
                continue          # drop with probability 1-easy_keep_prob
        out.append(h)
        if len(out) >= max_hits:
            break
    return out


# Build prompt_hits. Structure: {split: {query_accession: [hit, ...]}}.
prompt_hits = {
    s: {q: select_for_prompt(hs, s) for q, hs in hits[s].items()}
    for s in splits
}

# Print post-downsampling identity distribution
for s in splits:
    best = np.array([h[0]["fident"] for h in prompt_hits[s].values() if h])
    frac = sum(1 for h in prompt_hits[s].values() if h) / max(len(raw[s]), 1)
    if best.size:
        qs = np.percentile(best, [25, 50, 75])
        print(f"{s:5s} after downsampling: {frac:5.1%} have >=1 hit; "
              f"fident p25/50/75 = " + " / ".join(f"{v:.2f}" for v in qs))

# Save downsampled data
hits_path = os.path.join(work_dir, "prompt_hits.json")
with open(hits_path, "w") as fh:
    json.dump(prompt_hits, fh)
print("saved", hits_path)

train after downsampling: 96.8% have >=1 hit; fident p25/50/75 = 0.50 / 0.67 / 0.87
eval  after downsampling: 98.9% have >=1 hit; fident p25/50/75 = 0.25 / 0.35 / 0.53
test  after downsampling: 99.0% have >=1 hit; fident p25/50/75 = 0.25 / 0.35 / 0.55
saved /content/drive/MyDrive/Colab_Notebooks/266_Final_Project/foldseek_work/prompt_hits.json


## Dataset and collater


In [ ]:
# This cell defines the Dataset and Collater classes that build training prompts from CSV data and Foldseek retrieval hits. The collater assembles
# Llama chat prompts augmented with structural retrieval, tokenises sequences for the ESM2 encoder, and returns the full batch tensors expected by the
# model.

system_message = (
    "You are a scientific assistant specialized in protein function "
    "predictions. Given the sequence embeddings and other information "
    "of a protein, describe its function clearly and concisely in "
    "professional language. "
)
placeholder_token = "<|reserved_special_token_1|>"   # id 128003

from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer


class Prot2TextFoldseekDataset(Dataset):
    """Row-wise csv access + attached foldseek hits. Mirrors Prot2TextLightDataset from prot2textv2 paper."""

    def __init__(self, df, hits):
        self.data = df.reset_index(drop=True)
        self.hits = hits

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        acc = row["AlphaFoldDB"]
        return {
            "accession": acc,
            "taxon": row["taxon"],
            "sequence": row["sequence"],
            "function": row["function"],
            "hits": self.hits.get(acc, []),
        }


class Prot2TextFoldseekCollater:
    """
    Builds the chat prompt with taxonomy + foldseek retrieval, no protein name. mode is "train" or "inference".
    include_retrieval=False reproduces a name-free baseline (taxonomy + sequence only) which is the control condition
    random_crop defaults to True in train and False at inference. Based on Prot2TextLightCollater from prot2textv2 paper
    """

    def __init__(
        self,
        sequence_tokenizer,
        description_tokenizer,
        mode,
        taxonomy_dropout,
        retrieval_dropout,
        include_retrieval,
        max_hits,
        max_hit_chars,
        max_sequence_length,
        max_description_length,
        taxon_lookup=None,
        random_crop=None,
        seed=0,
    ):
        assert mode in ("train", "inference"), f"bad mode: {mode!r}"
        self.st = sequence_tokenizer
        self.dt = description_tokenizer
        self.mode = mode
        self.taxonomy_dropout = taxonomy_dropout
        self.retrieval_dropout = retrieval_dropout
        self.include_retrieval = include_retrieval
        self.max_hits = max_hits
        self.max_hit_chars = max_hit_chars
        self.max_sequence_length = max_sequence_length
        self.max_description_length = max_description_length
        self.taxon_lookup = taxon_lookup or {}    # accession -> organism string
        self.random_crop = (mode == "train") if random_crop is None else random_crop
        self.rng = random.Random(seed)

    # retrieval serialization
    def format_hits(self, hits):
      '''Serialise the Foldseek retrieval hits into a single text string for insertion into the prompt. Returns one of three things:
         None- omit the retrieval segment entirely (ablation or dropout)
         "none found"- search ran but found no usable hits for this protein
         string- formatted hit list with statistics and descriptions'''
        if not self.include_retrieval:
            return None
        if self.mode == "train" and self.rng.random() < self.retrieval_dropout:
            return None
        if not hits:
            return "none found"
        parts = []
        # Iterate over the top hits and truncate to avoid overly long prompt length
        for i, h in enumerate(hits[: self.max_hits], 1):
            d = str(h["desc"])[: self.max_hit_chars]
            if len(str(h["desc"])) > self.max_hit_chars:
                d = d.rsplit(" ", 1)[0].rstrip(".,;: ") + "..."
            d = " ".join(d.split())
            hit_org = self.taxon_lookup.get(h["target"], "unknown organism")
            parts.append(
                f"({i}) {h['fident']*100:.0f}% id, {h['qcov']:.2f} cov, "
                f"E={h['evalue']:.0e}, org={hit_org}: {d}"
            )
        return " ".join(parts)

    def build_user_message(self, taxon, hits, n_placeholders):
      '''Assemble the user message string passed to the Llama chat template.
        The prompt has three segments joined by " ; ":
        Taxon: {organism}
        Similar structures: {hit block}    <- omitted if format_hits returns None
        Sequence embeddings: <ph><ph>...   <- one placeholder per encoder token
      '''
        seg = [f"Taxon: {taxon}"]
        block = self.format_hits(hits)
        if block is not None:
            seg.append(f"Similar structures: {block}")
        seg.append("Sequence embeddings: " + placeholder_token * n_placeholders)
        return " ; ".join(seg)

    def __call__(self, batch):
      '''Collate a list of dataset items into model-ready tensors. Called automatically by pytorch dataload on each batch. Applies
        taxonomy dropout, crops long sequences, builds retrieval-augmented prompts, tokenises all text inputs, and assembles the full batch dict expected by
        Esm2LlamaInstructForCausalLM. Padding follows same method as prot2textv2 paper for esm2 input and llama decoder input
          '''
        accessions = [b["accession"] for b in batch]
        descriptions = [b["function"] if isinstance(b["function"], str) else "" for b in batch]

        taxons = [
            b["taxon"]
            if isinstance(b["taxon"], str) and self.rng.random() > self.taxonomy_dropout
            else "unknown"
            for b in batch
        ]

        # truncate long protein sequences
        seqs = []
        for b in batch:
            s = b["sequence"] if isinstance(b["sequence"], str) else ""
            if len(s) > self.max_sequence_length:
                start = (self.rng.randint(0, len(s) - self.max_sequence_length)
                         if self.random_crop else 0)
                s = s[start : start + self.max_sequence_length]
            seqs.append(s)
        #add padding to esm2 tokenizer input
        self.st.padding_side = "right"
        tok_seq = self.st(
            seqs, truncation=True, padding="longest",
            max_length=self.max_sequence_length + 2, return_tensors="pt",
        )
        seq_ids, seq_mask = tok_seq["input_ids"], tok_seq["attention_mask"]
        seq_lens = seq_mask.sum(dim=1).tolist()   # placeholders must match this exactly

        user_messages = [
            self.build_user_message(t, b["hits"], n)
            for t, b, n in zip(taxons, batch, seq_lens)
        ]
        convos = [
            [{"role": "system", "content": system_message},
             {"role": "user", "content": um}]
            for um in user_messages
        ]
        #add padding to llama decoder input
        self.dt.padding_side = "left"
        tok_prompt = self.dt.apply_chat_template(
            convos, add_generation_prompt=True, tokenize=True,
            padding="longest", return_tensors="pt", return_dict=True,
        )
        prompt_ids, prompt_mask = tok_prompt["input_ids"], tok_prompt["attention_mask"]
        #add padding to protein function description
        self.dt.padding_side = "right"
        tok_desc = self.dt(
            [d + self.dt.eos_token for d in descriptions],
            add_special_tokens=False, truncation=True, padding="longest",
            max_length=self.max_description_length, return_tensors="pt",
        )
        desc_ids, desc_mask = tok_desc["input_ids"], tok_desc["attention_mask"]

        labels = desc_ids.clone()
        labels[desc_mask == 0] = -100

        out = {
            "accession": accessions,
            "protein_input_ids": seq_ids,
            "protein_attention_mask": seq_mask,
            "description_input_ids": desc_ids,
            "description_attention_mask": desc_mask,
        }
        if self.mode == "train":
            out.update({
                "input_ids": torch.cat([prompt_ids, desc_ids], dim=1),
                "attention_mask": torch.cat([prompt_mask, desc_mask], dim=1),
                "labels": torch.cat([torch.full_like(prompt_ids, -100), labels], dim=1),
            })
        else:
            out.update({"input_ids": prompt_ids, "attention_mask": prompt_mask})
        return out

In [ ]:
taxonomy_dropout       = 0.1
retrieval_dropout      = 0.1
#Max characters per retrieved description before truncation
max_hit_chars          = 200
# Max amino acid sequence length fed to ESM2
max_sequence_length    = 1021
# Max description length in tokens.
max_description_length = 512
trim_train             = 100000
trim_eval              = 500
trim_test              = None

esm_tokenizer = AutoTokenizer.from_pretrained(esm_path)
llama_tokenizer = AutoTokenizer.from_pretrained(
    llama_path, pad_token="<|reserved_special_token_0|>"
)
# Verify the placeholder token maps to id 128003.
placeholder_id = llama_tokenizer.convert_tokens_to_ids(placeholder_token)
print("placeholder id:", placeholder_id, "(config default is 128003)")
print("pad id        :", llama_tokenizer.pad_token_id, "(generation uses 128002)")
assert placeholder_id == 128003, "placeholder id must match Esm2LlamaInstructConfig.placeholder_id"


def make_split(split, mode, include_retrieval=True,
               taxonomy_dropout=taxonomy_dropout,
               retrieval_dropout=retrieval_dropout,
               trim=None):
    # Creates a matched (dataset, collater) pair for one split.
    df = raw[split].sample(n=trim, random_state=random_seed) if trim else raw[split]
    ds = Prot2TextFoldseekDataset(df, prompt_hits[split])
    col = Prot2TextFoldseekCollater(
        sequence_tokenizer=esm_tokenizer,
        description_tokenizer=llama_tokenizer,
        mode=mode,
        taxonomy_dropout=taxonomy_dropout,
        retrieval_dropout=retrieval_dropout,
        include_retrieval=include_retrieval,
        max_hits=max_hits_in_prompt,
        max_hit_chars=max_hit_chars,
        max_sequence_length=max_sequence_length,
        max_description_length=max_description_length,
        taxon_lookup=hit_taxon_lookup,
        seed=random_seed,
    )
    return ds, col


# Create the training and evaluation pairs used by the dataloaders
train_ds, train_col = make_split("train", "train", trim=trim_train)
eval_ds,  eval_col  = make_split("eval",  "train", trim=trim_eval)
print(f"train {len(train_ds):,} | eval {len(eval_ds):,}")

placeholder id: 128003 (config default is 128003)
pad id        : 128002 (generation uses 128002)
train 100,000 | eval 500


In [ ]:
# --- sanity check: read an actual prompt before spending a GPU-hour on it -------
probe_ds, probe_col = make_split("test", "inference", taxonomy_dropout=0.0,
                                   retrieval_dropout=0.0, trim=8)
b_cols = probe_col([probe_ds[i] for i in range(4)])
b_text = llama_tokenizer.decode(b_cols["input_ids"][0], skip_special_tokens=False)

# collapse the placeholder run so the structure is readable
b_text = re.sub(r"(?:" + re.escape(placeholder_token) + r"){2,}",
               f"{placeholder_token}...x{int((_b['input_ids'][0]==placeholder_id).sum())}", _text)
print(_text)
print("\n--- shapes ---")
for k, v in _b.items():
    if torch.is_tensor(v): print(f"{k:28s} {tuple(v.shape)}")

# placeholder count must equal encoder valid positions, per sample
n_placeholder = (b_cols["input_ids"] == placeholder_id).sum(dim=1)
n_encoder = b_cols["protein_attention_mask"].sum(dim=1)
assert torch.equal(n_placeholder, n_encoder), f"placeholder/encoder mismatch: {n_placeholder} vs {n_encoder}"
print("\nplaceholder count matches encoder mask:", n_placeholder.tolist())

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

You are a scientific assistant specialized in protein function predictions. Given the sequence embeddings and other information of a protein, describe its function clearly and concisely in professional language.<|eot_id|><|start_header_id|>user<|end_header_id|>

Taxon: Saccharomyces ; Similar structures: (1) 34% id, 0.90 cov, E=2e-29, org=Aspergillus subgen. Fumigati: MFS transporter involved in the basal level of azole susceptibility. Confers resistance to voriconazole and, to a lesser extent, to fluconazole. (2) 32% id, 0.91 cov, E=3e-29, org=Fusarium fujikuroi species complex: Efflux pump; part of the gene cluster that mediates the biosynthesis of bikaverin, a red pigment also considered as a mycotoxin. ; Sequence embeddings: <|reserved_special_token_1|>...x516<|eot_id|><|start_header_id|>assistant<|end_header_id|>



--- shapes ---
protein_input_ids           

## Model Build


In [ ]:
init_from = "hf"

import importlib, importlib.util, types
sys.path.insert(0, repo_root)

def import_model_classes():
    """
    Import Esm2LlamaInstructForCausalLM, ModalityAdapter, and ModalityAdapterConfig from the repo clone
    """

    from models import ModalityAdapter, ModalityAdapterConfig, Esm2LlamaInstructForCausalLM
    print("imported via models/__init__.py")
    return ModalityAdapter, ModalityAdapterConfig, Esm2LlamaInstructForCausalLM

    mm = sys.modules["p2t_models.modeling_esm2llama_instruct"]
    cm = sys.modules["p2t_models.configuration_esm2llama_instruct"]
    return mm.ModalityAdapter, cm.ModalityAdapterConfig, mm.Esm2LlamaInstructForCausalLM

ModalityAdapter = ModalityAdapterConfig = Esm2LlamaInstructForCausalLM = None
ModalityAdapter, ModalityAdapterConfig, Esm2LlamaInstructForCausalLM = import_model_classes()


imported via models/__init__.py


In [ ]:
from huggingface_hub import snapshot_download
from safetensors.torch import load_file
from peft import get_peft_model, LoraConfig
from peft.peft_model import PeftModel
from transformers import EsmModel, EsmConfig, LlamaForCausalLM, LlamaConfig
from huggingface_hub import snapshot_download
from safetensors.torch import load_file

torch_dtype                 = torch.bfloat16
lora_rank                   = 16
lora_alpha                  = 32
lora_dropout                = 0.1
# Freeze modality projector aligned in stage-1 H-scale pretraining if True
fix_modality_adapter        = True
gradient_checkpointing      = True
# Only used for init_from='components'. Matches the repo default.
adapter_intermediate_dim    = 2048
# Paths used only for init_from='components'. Empty strings for 'hf' path.
load_model_checkpoint_path  = ""
load_adapter_checkpoint_dir = ""

# The Llama decoder weight matrices targeted by LORA.
lora_targets = [
    "self_attn.q_proj", "self_attn.k_proj", "self_attn.v_proj",
    "self_attn.o_proj", "mlp.gate_proj", "mlp.up_proj", "mlp.down_proj",
]


def build_base_from_hf():
  '''Builds the base pro2textv2 model from the HF checkpoint
  '''
    # Load released prot2textv2 weights into the model class
    print("downloading weights for:", hf_model_id)
    local_dir = snapshot_download(
        hf_model_id,
        allow_patterns=["*.json", "*.safetensors"],
    )

    with open(os.path.join(local_dir, "config.json")) as fh:
        hf_config = json.load(fh)

    # Reconstruct each sub-model's config from the nested dicts in config.json
    esm_config = EsmConfig(**hf_config["esm_config"])
    adapter_config = ModalityAdapterConfig(**hf_config["adapter_config"])
    llama_config = LlamaConfig(**hf_config["llama_config"])
    placeholder_id = hf_config.get("placeholder_id", 128003)

    # Instantiate the architecture with random weights initially
    esm_encoder = EsmModel(esm_config, add_pooling_layer=False)
    adapter = ModalityAdapter(adapter_config)
    llama_decoder = LlamaForCausalLM(llama_config)
    base = Esm2LlamaInstructForCausalLM(
        esm_encoder=esm_encoder, adapter=adapter, llama_decoder=llama_decoder,
        placeholder_id=placeholder_id,
    )

    # Detect sharded vs. single-file checkpoint and load all shards.
    index_path = os.path.join(local_dir, "model.safetensors.index.json")
    state_dict = {}
    if os.path.exists(index_path):
        with open(index_path) as fh:
            shard_files = sorted(set(json.load(fh)["weight_map"].values()))
    else:
        shard_files = [f for f in os.listdir(local_dir) if f.endswith(".safetensors")]
    for shard in shard_files:
        state_dict.update(load_file(os.path.join(local_dir, shard)))

    missing, unexpected = base.load_state_dict(state_dict, strict=False)
    print(f"loaded {len(state_dict)} tensors  missing={len(missing)} unexpected={len(unexpected)}")
    if missing:
        print("  e.g. missing:", missing[:5])
    if unexpected:
        print("  e.g. unexpected:", unexpected[:5])
    assert not missing, "state dict did not fully populate the local architecture - check config keys"

    # Confirm the projector weights are present and trained (non-zero mean).
    sd = base.state_dict()
    assert any(k.startswith("adapter.fc1") for k in sd), \
        "no adapter.fc1 in released weights - check the checkpoint"
    w = sd["adapter.fc1.weight"].float()
    print(f"adapter.fc1 {tuple(w.shape)}  |w| mean={w.abs().mean():.4f} std={w.std():.4f}")
    print("placeholder_id:", base.config.placeholder_id)
    return base.to(torch_dtype)

def load_model(trainable=True):
    # Build the base model from init_from, then attach a fresh LoRA adapter.
    base = build_base_from_hf()

    print(f"initializing fresh LoRA (r={lora_rank})")
    m = get_peft_model(base, LoraConfig(
        r=lora_rank,
        lora_alpha=lora_alpha,
        lora_dropout=lora_dropout,
        bias="none",
        init_lora_weights=True,      # A random, B zero - standard LoRA init
        target_modules=lora_targets,
        # modules_to_save=None means projector stays frozen.
        modules_to_save=(None if fix_modality_adapter
                          else ["adapter.fc1", "adapter.fc2"]),
    ))

    m.print_trainable_parameters()

    hit = [n for n, _ in m.named_modules() if "lora_A" in n]
    print(f"LoRA on {len(hit)} decoder modules")
    return m


model = load_model(trainable=True).to("cuda")

if gradient_checkpointing:
  #allows for gradient checkpointing to conserve memory. Unwraps peft layer to get to Esm2LlamaInstructForCausalLM and esm_encoder/llama_decoder layers
    model.train()
    base_mod = model.base_model.model if hasattr(model, "base_model") else model
    gc_kwargs = {"gradient_checkpointing_kwargs": {"use_reentrant": False}}
    base_mod.llama_decoder.gradient_checkpointing_enable(**gc_kwargs)
    base_mod.esm_encoder.gradient_checkpointing_enable(**gc_kwargs)
    base_mod.llama_decoder.config.use_cache = False
    print("gradient checkpointing enabled (non-reentrant)")

print(f"\nallocated: {torch.cuda.memory_allocated()/1e9:.1f} GB")

downloading weights for: xiao-fei/Prot2Text-V2-11B-Instruct-hf


Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

loaded 917 tensors  missing=0 unexpected=1
  e.g. unexpected: ['esm_encoder.embeddings.position_embeddings.weight']
adapter.fc1 (2048, 2560)  |w| mean=0.1336 std=0.1685
placeholder_id: 128003
initializing fresh LoRA (r=16)
trainable params: 41,943,040 || all params: 10,918,298,529 || trainable%: 0.38415362877828857
LoRA on 448 decoder modules
gradient checkpointing enabled (non-reentrant)

allocated: 21.8 GB


## Training


In [ ]:
from torch.optim import Adam
from tqdm.auto import tqdm
num_epochs                  = 1
# Number of micro-batches whose gradients accumulate before each optimizer step.
gradient_accumulation_steps = 16
# Max gradient norm for clipping
gradient_clipping           = 1.0




def forward_loss(model, batch):
    # Forward pass returning cross-entropy loss over description tokens.
    return model(
        input_ids=batch["input_ids"].cuda(),
        attention_mask=batch["attention_mask"].cuda(),
        labels=batch["labels"].cuda(),
        protein_input_ids=batch["protein_input_ids"].cuda(),
        protein_attention_mask=batch["protein_attention_mask"].cuda(),
        output_attentions=False,
        output_hidden_states=False, return_dict=False,
    )[0]


def train_epoch(model, loader, optimizer, epoch):
  '''Training epoch with gradient accumulation.The loss is divided by gradient_accumulation_steps before backward() sothat accumulated gradients match the scale of a single large batch.
     Gradient clipping is applied after accumulation but before the optimizer step. Returns mean training loss over all micro-batches in the epoch.
  '''

    model.train()
    acc_loss, n_batch, acc_gn, n_step = 0.0, 0, 0.0, 0
    optimizer.zero_grad(set_to_none=True)
    t = tqdm(loader, desc=f"train {epoch}/{num_epochs}")
    for i, batch in enumerate(t):
        # Scale loss before backward so gradients accumulate at the correct
        # magnitude for the effective batch size.
        loss = forward_loss(model, batch) / gradient_accumulation_steps
        loss.backward()
        acc_loss += loss.item() * gradient_accumulation_steps
        n_batch += 1
        if (i + 1) % gradient_accumulation_steps == 0:
            # After accumulating the target number of micro-batches: clip and step.
            gn = torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad],
                max_norm=float("inf") if gradient_clipping is None else gradient_clipping,
            )
            acc_gn += float(gn); n_step += 1
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)
        t.set_postfix(loss=f"{acc_loss/max(n_batch,1):.4f}")
    mean_loss = acc_loss / max(n_batch, 1)
    if mean_loss != mean_loss:
        raise ValueError("NaN in training loss")
    print(f"[epoch {epoch}] train_loss={mean_loss:.4f} "
          f"lr={optimizer.param_groups[0]['lr']:.2e} gradnorm={acc_gn/max(n_step,1):.3f}")
    return mean_loss


@torch.no_grad()
def eval_epoch(model, loader, epoch):
    # Compute mean loss over the evaluation set without updating weights. @torch.no_grad() disables gradient tracking for efficiency.
    model.eval()
    acc, n = 0.0, 0
    for batch in tqdm(loader, desc=f"eval {epoch}/{num_epochs}"):
        acc += forward_loss(model, batch).item(); n += 1
    mean_loss = acc / max(n, 1)
    print(f"[epoch {epoch}] eval_loss={mean_loss:.4f}")
    return mean_loss

In [ ]:
# Number of proteins per micro-batch
batch_size  = 4
# dataloader worker processes for parallel data loading and tokenisation.
num_workers = 4

# Create training dataloader
train_loader = DataLoader(
    train_ds, batch_size=batch_size, shuffle=True, collate_fn=train_col,
    num_workers=num_workers, pin_memory=True, drop_last=True)

# Create evaluation dataloader
eval_loader = DataLoader(
    eval_ds, batch_size=batch_size, shuffle=False, collate_fn=eval_col,
    num_workers=num_workers, pin_memory=True, drop_last=True)

print(f"train micro-batches/epoch : {len(train_loader):,}")
print(f"optimizer steps/epoch     : {len(train_loader)//gradient_accumulation_steps:,}")
print(f"eval micro-batches/epoch  : {len(eval_loader):,}")

train micro-batches/epoch : 25,000
optimizer steps/epoch     : 1,562
eval micro-batches/epoch  : 125


In [ ]:
#cell rerun after error- training and eval were performed but are commented out to rerun cell
learning_rate   = 1e-4


optimizer = Adam([p for p in model.parameters() if p.requires_grad], lr=learning_rate)

history = []
run_start = time.time()
for epoch in range(1, num_epochs + 1):
    # tr = train_epoch(model, train_loader, optimizer, epoch)
    # ev = eval_epoch(model, eval_loader, epoch)
    history.append({"epoch": epoch, "train_loss": tr, "eval_loss": ev,
                    "elapsed_s": time.time() - run_start})

    ckpt = os.path.join(out_dir, f"adapter_checkpoint_{epoch}")
    model.save_pretrained(ckpt)
    torch.save({"optimizer_state_dict": optimizer.state_dict()},
               os.path.join(out_dir, f"optimizer_scheduler_checkpoint_{epoch}.pt"))
    print(f"saved {ckpt}  (elapsed {(time.time()-run_start)/3600:.2f} h)")

print(f"\nactual total: {(time.time()-run_start)/3600:.2f} h  "
      f"| estimated: {est['total_s']/3600:.2f} h")
with open(os.path.join(out_dir, "history.json"), "w") as fh:
    json.dump(history, fh, indent=2)
history

/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:154: UserWarning: Could not find a config file in  - will assume that the vocabulary was not modified.
  warnings.warn(


saved /content/drive/MyDrive/Colab_Notebooks/266_Final_Project/foldseek_lora/drop01run/adapter_checkpoint_1  (elapsed 0.00 h)

actual total: 0.00 h  | estimated: 9.29 h


[{'epoch': 1,
  'train_loss': 0.010711784770367059,
  'eval_loss': 0.897360550545156,
  'elapsed_s': 0.0003025531768798828}]

## Prediction Generation

In [ ]:
gen_batch_size        = 4
# Max new tokens to generate per protein
max_generation_length = 256
# Greedy decoding
num_beams             = 1
do_sample             = False
# <|eot_id|>: the token llama uses to end an assistant turn.
gen_eos_token_id      = 128009
# reserved_special_token_0
gen_pad_token_id      = 128002
gen_num_workers       = 2


@torch.no_grad()
def generate_split(model, split, include_retrieval, tag, trim=None):
  '''Generate function descriptions for every protein in one split and save results as a JSON file. At inference, all dropout is disabled so every
     protein receives its full prompt. drop_last=False ensures no proteins are silently discarded. Saves {accession: {true, pred}} to
     out_dir/generation_{tag}.json and returns the same dict.
  '''

    ds, col = make_split(split, "inference", include_retrieval=include_retrieval,
                         taxonomy_dropout=0.0, retrieval_dropout=0.0, trim=trim)
    loader = DataLoader(ds, batch_size=gen_batch_size, shuffle=False, collate_fn=col,
                        num_workers=gen_num_workers, pin_memory=True, drop_last=False)
    model.eval()
    results = {}
    for batch in tqdm(loader, desc=f"generate [{tag}]"):
        out = model.generate(
            inputs=batch["input_ids"].cuda(),
            attention_mask=batch["attention_mask"].cuda(),
            protein_input_ids=batch["protein_input_ids"].cuda(),
            protein_attention_mask=batch["protein_attention_mask"].cuda(),
            max_new_tokens=max_generation_length,
            eos_token_id=gen_eos_token_id,
            pad_token_id=gen_pad_token_id,
            return_dict_in_generate=False,
            num_beams=num_beams,
            do_sample=do_sample,
        )
        preds = llama_tokenizer.batch_decode(out.cpu(), skip_special_tokens=True)
        refs = llama_tokenizer.batch_decode(batch["description_input_ids"], skip_special_tokens=True)
        for acc, ref, pred in zip(batch["accession"], refs, preds):
            results[acc] = {"true": ref.strip(), "pred": pred.strip()}
    path = os.path.join(out_dir, f"generation_{tag}.json")
    with open(path, "w") as fh:
        json.dump(results, fh, indent=2)
    print(f"saved {path}  ({len(results):,} proteins)")
    return results

In [ ]:
# Merge LORA into the base weights for faster, deterministic inference.
gen_model = model.merge_and_unload()
gen_model.eval()
if hasattr(gen_model, "llama_decoder"):
    gen_model.llama_decoder.config.use_cache = True
#Generate predictions with and without the foldseek retrieval
gen_with    = generate_split(gen_model, "test", include_retrieval =True,  "test_retrieval",   TRIM_TEST)
gen_without = generate_split(gen_model, "test", include_retrieval =False, "test_noretrieval", TRIM_TEST)

for k in list(gen_with)[:2]:
    print("=" * 70)
    print("ACC :", k)
    print("TRUE:", gen_with[k]["true"][:400])
    print("PRED:", gen_with[k]["pred"][:400])

generate [test_retrieval]:   0%|          | 0/1051 [00:00<?, ?it/s]

saved /content/drive/MyDrive/Colab_Notebooks/266_Final_Project/foldseek_lora/drop01run/generation_test_retrieval.json  (4,203 proteins)


generate [test_noretrieval]:   0%|          | 0/1051 [00:00<?, ?it/s]

saved /content/drive/MyDrive/Colab_Notebooks/266_Final_Project/foldseek_lora/drop01run/generation_test_noretrieval.json  (4,203 proteins)
ACC : Q9W3K5
TRUE: Catalyzes the ATP-dependent ligation of L-glutamate and L-cysteine and participates in the first and rate-limiting step in glutathione biosynthesis.
PRED: Catalyzes the ATP-dependent ligation of L-glutamate and L-cysteine and participates in the first and rate-limiting step in glutathione biosynthesis.
ACC : Q9CWH0
TRUE: Binds to target gene promoters, including NKX2-5 and SYCE1, but not GATA4, and may be involved in the maintenance of the active epigenetic status of these genes.
PRED: Binds to the promoter of target genes and plays a role in the regulation of transcription.


## Evaluation

In [ ]:
#The functions are lifted mainly from the prot2txtv2 paper
# Max tokens fed to the BERTscore model
bert_truncate_length = 495
roberta_model        = "FacebookAI/roberta-large"
biobert_model        = "dmis-lab/biobert-large-cased-v1.1"
biobert_num_layers   = 24

import evaluate
from transformers import BertTokenizer, RobertaTokenizer


def compute_exact_match(predictions, references):
    # Fraction of predictions that exactly match their reference after
    # lowercasing and stripping all non-word characters. A very strict metric;
    # non-zero scores indicate verbatim agreement.
    def normalize(t):
        return re.sub(r"[^\w]", "", t.lower())
    return sum(normalize(p) == normalize(r) for p, r in zip(predictions, references)) / len(predictions)


def compute_bleu(predictions, references, max_order=4):
    # BLEU score measuring n-gram precision up to max_order.
    # BLEU-2 and BLEU-4 are both reported in the Prot2Text-V2 paper.
    return evaluate.load("bleu").compute(
        predictions=predictions, references=references, max_order=max_order)


def compute_rouge(predictions, references):
    # ROUGE scores measuring recall-oriented n-gram and subsequence overlap.
    # Returns rouge1 (unigram), rouge2 (bigram), and rougeL (LCS-based).
    return evaluate.load("rouge").compute(predictions=predictions, references=references)


def _truncate(texts, tokenizer, max_length=bert_truncate_length):
    # Tokenise and decode texts to fit within the BERTScore model's limit.
    ids = tokenizer(texts, padding="max_length", truncation=True,
                    max_length=max_length, return_tensors="pt")["input_ids"]
    return tokenizer.batch_decode(ids, skip_special_tokens=True)


def compute_bert_score(predictions, references):
    # BERTScore using both RoBERTa-large and BioBERT-large. BERTScore computes
    # token-level cosine similarities between contextual embeddings of prediction
    # and reference tokens, aggregated into precision, recall, and F1.
    bert, results = evaluate.load("bertscore"), {}

    rt = RobertaTokenizer.from_pretrained(roberta_model)
    r = bert.compute(predictions=_truncate(predictions, rt),
                     references=_truncate(references, rt), lang="en")
    results["roberta-large"] = {k: sum(r[k]) / len(r[k]) for k in ("precision", "recall", "f1")}

    bt = BertTokenizer.from_pretrained(biobert_model)
    b = bert.compute(predictions=_truncate(predictions, bt),
                     references=_truncate(references, bt),
                     model_type=biobert_model, num_layers=biobert_num_layers)
    results["biobert-large"] = {k: sum(b[k]) / len(b[k]) for k in ("precision", "recall", "f1")}
    return results


def compute_metrics(predictions, references, bert_score=True, verbose=True):
    # Compute the full metric suite: exact match, BLEU-2, BLEU-4, ROUGE, and
    # BERTScore. bert_score=False skips the expensive BERTScore computation
    # (used for stratified bin analysis where it would be called many times).
    out = {"n": len(predictions)}
    out["exact_match"] = compute_exact_match(predictions, references)
    out["bleu2"] = compute_bleu(predictions, references, max_order=2)
    out["bleu4"] = compute_bleu(predictions, references, max_order=4)
    out["rouge"] = compute_rouge(predictions, references)
    if bert_score:
        out["bert"] = compute_bert_score(predictions, references)
    if verbose:
        print(f"n            : {out['n']}")
        print(f"exact match  : {out['exact_match']:.4f}")
        print(f"BLEU-2       : {out['bleu2']['bleu']:.4f}")
        print(f"BLEU-4       : {out['bleu4']['bleu']:.4f}")
        print(f"ROUGE-1/2/L  : {out['rouge']['rouge1']:.4f} / "
              f"{out['rouge']['rouge2']:.4f} / {out['rouge']['rougeL']:.4f}")
        if bert_score:
            for m, v in out["bert"].items():
                print(f"BERTScore {m:14s} P/R/F1: {v['precision']:.4f} / "
                      f"{v['recall']:.4f} / {v['f1']:.4f}")
    return out


def score_results(results, **kw):
    # Helper that calls compute_metrics on a generation result dict.
    accs = list(results.keys())
    return accs, compute_metrics([results[a]["pred"] for a in accs],
                                 [results[a]["true"] for a in accs], **kw)

In [ ]:
# Compute all metrics on both generation conditions and print a comparison.
print("### WITH Foldseek retrieval ###")
_, m_with = score_results(gen_with)
print("\n### WITHOUT retrieval (taxonomy + sequence only) ###")
_, m_without = score_results(gen_without)

with open(os.path.join(OUT_DIR, "metrics.json"), "w") as fh:
    json.dump({"with_retrieval": m_with, "without_retrieval": m_without}, fh, indent=2)

print("\n" + "=" * 62)
print(f"{'metric':<16}{'with':>14}{'without':>14}{'delta':>14}")
print("-" * 62)
for label, get in [
    ("exact match", lambda m: m["exact_match"]),
    ("BLEU-2",      lambda m: m["bleu2"]["bleu"]),
    ("BLEU-4",      lambda m: m["bleu4"]["bleu"]),
    ("ROUGE-1",     lambda m: m["rouge"]["rouge1"]),
    ("ROUGE-L",     lambda m: m["rouge"]["rougeL"]),
    ("BERT rob F1", lambda m: m["bert"]["roberta-large"]["f1"]),
    ("BERT bio F1", lambda m: m["bert"]["biobert-large"]["f1"]),
]:
    a, b = get(m_with), get(m_without)
    print(f"{label:<16}{a:>14.4f}{b:>14.4f}{a-b:>+14.4f}")

### WITH Foldseek retrieval ###


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/1.46G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.46G [00:00<?, ?B/s]

n            : 4203
exact match  : 0.3685
BLEU-2       : 0.3829
BLEU-4       : 0.3478
ROUGE-1/2/L  : 0.5420 / 0.4706 / 0.5234
BERTScore roberta-large  P/R/F1: 0.9167 / 0.9099 / 0.9129
BERTScore biobert-large  P/R/F1: 0.8597 / 0.8523 / 0.8551

### WITHOUT retrieval (taxonomy + sequence only) ###


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


n            : 4203
exact match  : 0.3605
BLEU-2       : 0.3757
BLEU-4       : 0.3405
ROUGE-1/2/L  : 0.5327 / 0.4608 / 0.5143
BERTScore roberta-large  P/R/F1: 0.9152 / 0.9083 / 0.9113
BERTScore biobert-large  P/R/F1: 0.8583 / 0.8497 / 0.8530

metric                    with       without         delta
--------------------------------------------------------------
exact match             0.3685        0.3605       +0.0081
BLEU-2                  0.3829        0.3757       +0.0072
BLEU-4                  0.3478        0.3405       +0.0073
ROUGE-1                 0.5420        0.5327       +0.0093
ROUGE-L                 0.5234        0.5143       +0.0092
BERT rob F1             0.9129        0.9113       +0.0015
BERT bio F1             0.8551        0.8530       +0.0021


In [ ]:
# Identity bins for stratified analysis. Test proteins are binned by best-hit sequence identity (fident).
ident_bins   = [(0.00, 0.20, "<20%"), (0.20, 0.30, "20-30%"),
                (0.30, 0.50, "30-50%"), (0.50, 1.01, ">50%")]
# Bins with fewer proteins than this are reported as count only
min_bin_size = 20
def best_fident(acc):
    # Return best-hit sequence identity for a test accession, or None.
    hs = prompt_hits["test"].get(acc, [])
    return hs[0]["fident"] if hs else None


def bin_of(acc):
    # Map a test accession to its identity bin label.
    f = best_fident(acc)
    if f is None:
        return "no hit"
    for lo, hi, name in ident_bins:
        if lo <= f < hi:
            return name
    return ident_bins[-1][2]


# Compute bleu, rouge and bert for each bin and both conditions
rows = []
for name in ["no hit"] + [b[2] for b in ident_bins]:
    accs = [a for a in gen_with if bin_of(a) == name]
    if len(accs) < min_bin_size:
        rows.append({"bin": name, "n": len(accs)})
        continue
    r = {"bin": name, "n": len(accs)}
    for tag, res in [("with", gen_with), ("without", gen_without)]:
        m = compute_metrics([res[a]["pred"] for a in accs],
                            [res[a]["true"] for a in accs],
                            bert_score=True, verbose=False)
        r[f"bleu4_{tag}"]   = m["bleu4"]["bleu"]
        r[f"rougeL_{tag}"]  = m["rouge"]["rougeL"]
        r[f"bert_rob_{tag}"] = m["bert"]["roberta-large"]["f1"]
        r[f"bert_bio_{tag}"] = m["bert"]["biobert-large"]["f1"]
    r["d_bleu4"]    = r["bleu4_with"]    - r["bleu4_without"]
    r["d_rougeL"]   = r["rougeL_with"]   - r["rougeL_without"]
    r["d_bert_rob"] = r["bert_rob_with"]  - r["bert_rob_without"]
    r["d_bert_bio"] = r["bert_bio_with"]  - r["bert_bio_without"]
    rows.append(r)

strat = pd.DataFrame(rows)
strat.to_csv(os.path.join(OUT_DIR, "stratified_metrics.csv"), index=False)
strat

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You sho

,bin,n,bleu4_with,rougeL_with,bert_rob_with,bert_bio_with,bleu4_without,rougeL_without,bert_rob_without,bert_bio_without,d_bleu4,d_rougeL,d_bert_rob,d_bert_bio
0,no hit,40,0.395998,0.446716,0.897848,0.826384,0.396508,0.447239,0.899325,0.827158,-0.000511,-0.000522,-0.001477,-0.000774
1,<20%,638,0.145881,0.256200,0.864414,0.769919,0.135399,0.251030,0.863327,0.769427,0.010483,0.005170,0.001088,0.000492
2,20-30%,886,0.237077,0.402386,0.890866,0.816175,0.239765,0.394840,0.889975,0.815830,-0.002688,0.007546,0.000891,0.000345
3,30-50%,1457,0.343832,0.553170,0.917925,0.865839,0.333573,0.542315,0.916003,0.862962,0.010259,0.010855,0.001922,0.002876
4,>50%,1182,0.554776,0.725654,0.949853,0.917890,0.544780,0.715038,0.947930,0.914618,0.009995,0.010615,0.001923,0.003272


In [ ]:
# Sequence-only baseline generation (no taxonomy, no structural retrieval)

# Build a collater that drops both taxonomy and retrieval. taxonomy_dropout=1.0 means self.rng.random() is always < 1.0, so taxon is
# always replaced with "unknown", making the taxon field uninformative
seq_only_col = Prot2TextFoldseekCollater(
    sequence_tokenizer=esm_tokenizer,
    description_tokenizer=llama_tokenizer,
    mode="inference",
    taxonomy_dropout=1.0,        # always "unknown" - taxon never shown
    retrieval_dropout=0.0,       # irrelevant since include_retrieval=False
    include_retrieval=False,     # no Similar structures block
    max_hits=max_hits_in_prompt,
    max_hit_chars=max_hit_chars,
    max_sequence_length=max_sequence_length,
    max_description_length=max_description_length,
    taxon_lookup=hit_taxon_lookup,
    seed=random_seed,
)

# Verify the prompt looks as expected before running the full generation.
probe_ds_seq, _ = make_split("test", "inference",
                              include_retrieval=False,
                              taxonomy_dropout=1.0, trim=4)
probe_batch = seq_only_col([probe_ds_seq[i] for i in range(2)])
probe_text = llama_tokenizer.decode(
    probe_batch["input_ids"][0], skip_special_tokens=False)
probe_text = re.sub(
    r"(?:" + re.escape(placeholder_token) + r"){2,}",
    f"{placeholder_token}...x"
    f"{int((_probe_batch['input_ids'][0] == placeholder_id).sum())}",
    probe_text,
)
print("sequence-only prompt (first test protein):")
print(probe_text)
print()

# Run generation on the full test split using the sequence-only collater.
@torch.no_grad()
def generate_seq_only(model, trim=None):
    ds, _ = make_split("test", "inference",
                       include_retrieval=False,
                       taxonomy_dropout=1.0,
                       trim=trim)
    loader = DataLoader(
        ds, batch_size=gen_batch_size, shuffle=False,
        collate_fn=seq_only_col,
        num_workers=gen_num_workers, pin_memory=True, drop_last=False,
    )
    model.eval()
    results = {}
    for batch in tqdm(loader, desc="generate [seq-only]"):
        out = model.generate(
            inputs=batch["input_ids"].cuda(),
            attention_mask=batch["attention_mask"].cuda(),
            protein_input_ids=batch["protein_input_ids"].cuda(),
            protein_attention_mask=batch["protein_attention_mask"].cuda(),
            max_new_tokens=max_generation_length,
            eos_token_id=gen_eos_token_id,
            pad_token_id=gen_pad_token_id,
            return_dict_in_generate=False,
            num_beams=num_beams,
            do_sample=do_sample,
        )
        preds = llama_tokenizer.batch_decode(out.cpu(), skip_special_tokens=True)
        refs  = llama_tokenizer.batch_decode(
            batch["description_input_ids"], skip_special_tokens=True)
        for acc, ref, pred in zip(batch["accession"], refs, preds):
            results[acc] = {"true": ref.strip(), "pred": pred.strip()}
    path = os.path.join(out_dir, "generation_seq_only.json")
    with open(path, "w") as fh:
        json.dump(results, fh, indent=2)
    print(f"saved {path}  ({len(results):,} proteins)")
    return results

gen_seq_only = generate_seq_only(gen_model, trim=trim_test)

# print first two predictions
for k in list(gen_seq_only)[:2]:
    print("=" * 70)
    print("ACC  :", k)
    print("TRUE :", gen_seq_only[k]["true"][:300])
    print("PRED :", gen_seq_only[k]["pred"][:300])

# Score and compare all three conditions
print("\n### SEQUENCE ONLY (no taxonomy, no retrieval) ###")
_, m_seq_only = score_results(gen_seq_only)

print("\n" + "=" * 72)
print(f"{'metric':<16}{'seq-only':>14}{'without':>14}{'with':>14}")
print("-" * 72)
for label, get in [
    ("exact match", lambda m: m["exact_match"]),
    ("BLEU-2",      lambda m: m["bleu2"]["bleu"]),
    ("BLEU-4",      lambda m: m["bleu4"]["bleu"]),
    ("ROUGE-1",     lambda m: m["rouge"]["rouge1"]),
    ("ROUGE-L",     lambda m: m["rouge"]["rougeL"]),
    ("BERT rob F1", lambda m: m["bert"]["roberta-large"]["f1"]),
    ("BERT bio F1", lambda m: m["bert"]["biobert-large"]["f1"]),
]:
    s  = get(m_seq_only)
    wo = get(m_without)
    wi = get(m_with)
    print(f"{label:<16}{s:>14.4f}{wo:>14.4f}{wi:>14.4f}")

with open(os.path.join(out_dir, "metrics_seq_only.json"), "w") as fh:
    json.dump(m_seq_only, fh, indent=2)


sequence-only prompt (first test protein):
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

You are a scientific assistant specialized in protein function predictions. Given the sequence embeddings and other information of a protein, describe its function clearly and concisely in professional language.<|eot_id|><|start_header_id|>user<|end_header_id|>

Taxon: unknown ; Sequence embeddings: <|reserved_special_token_1|>...x516<|eot_id|><|start_header_id|>assistant<|end_header_id|>





generate [seq-only]:   0%|          | 0/1051 [00:00<?, ?it/s]

saved /content/drive/MyDrive/Colab_Notebooks/266_Final_Project/foldseek_lora/drop01run/generation_seq_only.json  (4,203 proteins)
ACC  : Q9W3K5
TRUE : Catalyzes the ATP-dependent ligation of L-glutamate and L-cysteine and participates in the first and rate-limiting step in glutathione biosynthesis.
PRED : Catalyzes the ATP-dependent ligation of L-glutamate and L-cysteine and participates in the first and rate-limiting step in glutathione biosynthesis.
ACC  : Q9CWH0
TRUE : Binds to target gene promoters, including NKX2-5 and SYCE1, but not GATA4, and may be involved in the maintenance of the active epigenetic status of these genes.
PRED : Binds to the promoter of target genes and plays a role in the regulation of transcription.

### SEQUENCE ONLY (no taxonomy, no retrieval) ###


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


n            : 4203
exact match  : 0.3583
BLEU-2       : 0.3712
BLEU-4       : 0.3355
ROUGE-1/2/L  : 0.5303 / 0.4578 / 0.5113
BERTScore roberta-large  P/R/F1: 0.9148 / 0.9078 / 0.9109
BERTScore biobert-large  P/R/F1: 0.8569 / 0.8487 / 0.8518

metric                seq-only       without          with
------------------------------------------------------------------------
exact match             0.3583        0.3605        0.3685
BLEU-2                  0.3712        0.3757        0.3829
BLEU-4                  0.3355        0.3405        0.3478
ROUGE-1                 0.5303        0.5327        0.5420
ROUGE-L                 0.5113        0.5143        0.5234
BERT rob F1             0.9109        0.9113        0.9129
BERT bio F1             0.8518        0.8530        0.8551
